In [1]:
import anndata as ad
import numpy as np
import pandas as pd
import zarr

print("anndata:", ad.__version__)
print("pandas:", pd.__version__)
print("zarr:", zarr.__version__)

obs = pd.DataFrame(
    {
        "np_nan_str": [np.nan, "cell1"],
        "np_nan_int": [np.nan, 1],
        "np_nan_float": [np.nan, 1.0],
        "pd_NA_str": [pd.NA, "cell1"],
        "pd_NA_int": [pd.NA, 1],
        "pd_NA_float": [pd.NA, 1.0],
    },
    index=["cell1", "cell2"],
)

adata = ad.AnnData(
    X=np.zeros((2, 1)),
    obs=obs,
    var=pd.DataFrame(index=["gene1"]),
)

print("Before writing")
print(adata.obs)
print(adata.obs.isna())

adata.write_zarr("toy_anndata.zarr")

reloaded = ad.read_zarr("toy_anndata.zarr")

print("\nAfter reading")
print(reloaded.obs)
print(reloaded.obs.isna())

print("\nDtypes")
print(reloaded.obs.dtypes)

/tmp/ipykernel_2447169/3418326256.py:6: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)
/home/lazic/.venvs/stpuppeteer_test/lib/python3.13/site-packages/anndata/_io/zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


anndata: 0.12.19
pandas: 2.3.3
zarr: 3.2.1
Before writing
      np_nan_str  np_nan_int  np_nan_float pd_NA_str pd_NA_int pd_NA_float
cell1        NaN         NaN           NaN      <NA>      <NA>        <NA>
cell2      cell1         1.0           1.0     cell1         1         1.0
       np_nan_str  np_nan_int  np_nan_float  pd_NA_str  pd_NA_int  pd_NA_float
cell1        True        True          True       True       True         True
cell2       False       False         False      False      False        False

After reading
      np_nan_str  np_nan_int  np_nan_float pd_NA_str pd_NA_int pd_NA_float
cell1        NaN         NaN           NaN       NaN      <NA>        <NA>
cell2      cell1         1.0           1.0     cell1         1         1.0
       np_nan_str  np_nan_int  np_nan_float  pd_NA_str  pd_NA_int  pd_NA_float
cell1        True        True          True       True      False        False
cell2       False       False         False      False      False        False

Dt

In [2]:
import anndata as ad
import geopandas as gpd
import numpy as np
import pandas as pd
import spatialdata as sd
import zarr
from shapely.geometry import Polygon
from spatialdata.models import ShapesModel, TableModel

print("spatialdata:", sd.__version__)
print("anndata:", ad.__version__)
print("pandas:", pd.__version__)
print("zarr:", zarr.__version__)

shapes = gpd.GeoDataFrame(
    {
        "geometry": [
            Polygon([(0, 0), (1, 0), (1, 1), (0, 1)]),
            Polygon([(2, 0), (3, 0), (3, 1), (2, 1)]),
        ]
    },
    index=["cell1", "cell2"],
)

obs = pd.DataFrame(
    {
        "instance_id": ["cell1", "cell2"],
        "region": ["cells", "cells"],
        "np_nan_str": [np.nan, "cell1"],
        "np_nan_int": [np.nan, 1],
        "np_nan_float": [np.nan, 1.0],
        "pd_NA_str": [pd.NA, "cell1"],
        "pd_NA_int": [pd.NA, 1],
        "pd_NA_float": [pd.NA, 1.0],
    },
    index=["cell1", "cell2"],
)

adata = ad.AnnData(
    X=np.zeros((2, 1)),
    obs=obs,
    var=pd.DataFrame(index=["gene1"]),
)

table = TableModel.parse(
    adata,
    region="cells",
    region_key="region",
    instance_key="instance_id",
)

sdata = sd.SpatialData(
    shapes={"cells": ShapesModel.parse(shapes)},
    tables={"counts": table},
)

print("\nBefore writing")
print(sdata.tables["counts"].obs)
print("\nMissing values before writing")
print(sdata.tables["counts"].obs.isna())
print("\nDtypes before writing")
print(sdata.tables["counts"].obs.dtypes)

sdata.write("toy_spatialdata.zarr", overwrite=True)

reloaded = sd.read_zarr("toy_spatialdata.zarr")

print("\nAfter reading")
print(reloaded.tables["counts"].obs)
print("\nMissing values after reading")
print(reloaded.tables["counts"].obs.isna())
print("\nDtypes after reading")
print(reloaded.tables["counts"].obs.dtypes)

spatialdata: 0.8.0
anndata: 0.12.19
pandas: 2.3.3
zarr: 3.2.1


/tmp/ipykernel_2447169/1155955862.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)
/home/lazic/.venvs/stpuppeteer_test/lib/python3.13/site-packages/spatialdata/models/models.py:1267: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)



Before writing
      instance_id region np_nan_str  np_nan_int  np_nan_float pd_NA_str  \
cell1       cell1  cells        NaN         NaN           NaN      <NA>   
cell2       cell2  cells      cell1         1.0           1.0     cell1   

      pd_NA_int pd_NA_float  
cell1      <NA>        <NA>  
cell2         1         1.0  

Missing values before writing
       instance_id  region  np_nan_str  np_nan_int  np_nan_float  pd_NA_str  \
cell1        False   False        True        True          True       True   
cell2        False   False       False       False         False      False   

       pd_NA_int  pd_NA_float  
cell1       True         True  
cell2      False        False  

Dtypes before writing
instance_id       object
region          category
np_nan_str        object
np_nan_int       float64
np_nan_float     float64
pd_NA_str         object
pd_NA_int         object
pd_NA_float       object
dtype: object


/home/lazic/.local/share/uv/python/cpython-3.13.5-linux-x86_64-gnu/lib/python3.13/contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/home/lazic/.local/share/uv/python/cpython-3.13.5-linux-x86_64-gnu/lib/python3.13/contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/home/lazic/.local/share/uv/python/cpython-3.13.5-linux-x86_64-gnu/lib/python3.13/contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/home/lazic/.local/share/uv/python/cpython-3.13.5-linux-x86_64-gnu/lib/python3.13/contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/home/lazic/.local/share/uv/python/cpython-3.13.5-linux-x86_64-gnu/lib/python3.13/contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor rel


After reading
      instance_id region np_nan_str  np_nan_int  np_nan_float pd_NA_str  \
cell1       cell1  cells        nan         NaN           NaN      <NA>   
cell2       cell2  cells      cell1         1.0           1.0     cell1   

      pd_NA_int pd_NA_float  
cell1      <NA>        <NA>  
cell2         1         1.0  

Missing values after reading
       instance_id  region  np_nan_str  np_nan_int  np_nan_float  pd_NA_str  \
cell1        False   False       False        True          True      False   
cell2        False   False       False       False         False      False   

       pd_NA_int  pd_NA_float  
cell1      False        False  
cell2      False        False  

Dtypes after reading
instance_id       object
region          category
np_nan_str        object
np_nan_int       float64
np_nan_float     float64
pd_NA_str         object
pd_NA_int         object
pd_NA_float       object
dtype: object
